# Raw Data Preparation: Satellite Bands and CAMS Integration

## Overview
This document outlines the preprocessing pipeline for the solar forecasting dataset. The workflow ingests combined 15-channel satellite imagery and CAMS (Copernicus Atmosphere Monitoring Service) data, segments the observations by diurnal cycles, and temporally aligns the datasets using fuzzy matching.

## 1. Initial Data Ingestion
The pipeline begins by loading the combined 15-channel satellite band data to verify its structure and temporal bounds.
* **Source File:** `Data/bands/combined_15ch_full.npz`
* **Data Dimensions:** `(154779, 15, 50, 50)`
* **End Date:** `2023-12-31 23:50:00`

## 2. Diurnal Segmentation
To account for differing solar and atmospheric conditions, the combined dataset is explicitly split into day and night/twilight subsets. Any timesteps that result entirely in `NaN` values during this split are dropped to maintain data integrity.
* **Daytime Output:** `combined_15ch_full_day_no_twillight.npz` *(Strict daylight, no twilight)*
* **Nighttime Output:** `combined_15ch_full_night_twillight.npz` *(Nighttime and twilight hours)*

## 3. Temporal Alignment
The segmented satellite bands are then paired with CAMS irradiance data (`ghi_grid2500_2ch_15mins.npz`, comprising 140,214 initial timestamps). Because sensor readings do not always perfectly align, a fuzzy matching algorithm is applied with a maximum tolerance of 6 minutes. One-to-many matching is permitted.

### Daytime Dataset Pairing
* **Input Source:** `combined_15ch_full_day_no_twillight.npz`
* **Matched Samples:** 67,638 successful pairs
* **Maximum Time Delta (|Δt|):** 5 minutes
* **Bands Output:** `Data/bands/modeldata/bands_day_paired_30mins_new_fuzzy_no_twillight.npz`
* **CAMS Output:** `Data/CAMS/modeldata/cams_day_paired_30mins_new_fuzzy_no_twillight.npz`

### Nighttime and Twilight Dataset Pairing
* **Input Source:** `combined_15ch_full_night_twillight.npz`
* **Matched Samples:** 87,016 successful pairs
* **Maximum Time Delta (|Δt|):** 5 minutes
* **Bands Output:** `Data/bands/modeldata/bands_night_paired_30mins_new_fuzzy_twillight.npz`
* **CAMS Output:** `Data/CAMS/modeldata/cams_night_paired_30mins_new_fuzzy_twillight.npz`

In [1]:
import numpy as np
%load_ext autoreload
%autoreload 2
from download.preproceessing import build_combined_npz,split_day_night_npz,pair_and_save_cams_bands

## 1. Initial Data Ingestion
The pipeline begins by loading the combined 15-channel satellite band data to verify its structure and temporal bounds.
* **Source File:** `Data/bands/combined_15ch_full.npz`
* **Data Dimensions:** `(154779, 15, 50, 50)`
* **End Date:** `2023-12-31 23:50:00`

In [ ]:
build_combined_npz(root="../Data/2kmhk", out="combined_15ch_full.npz")

In [9]:
z = np.load("Data/bands/combined_15ch_full.npz")
z.keys()

KeysView(NpzFile 'Data/bands/combined_15ch_full.npz' with keys: data, time_min)

In [10]:
z['time_min']

array([26824320, 26824330, 26824340, ..., 28401090, 28401100, 28401110],
      shape=(154779,))

Start Time: 26824320 minutes → 2021-01-01 00:00:00 UTC
End Time: 28401110 minutes → 2023-12-31 23:50:00 UTC

## 2. Diurnal Segmentation
To account for differing solar and atmospheric conditions, the combined dataset is explicitly split into day and night/twilight subsets. Any timesteps that result entirely in `NaN` values during this split are dropped to maintain data integrity.
* **Daytime Output:** `combined_15ch_full_day_no_twillight.npz` *(Strict daylight, no twilight)*
* **Nighttime Output:** `combined_15ch_full_night_twillight.npz` *(Nighttime and twilight hours)*

In [ ]:
split_day_night_npz(
    "../Data/bands/combined_15ch_full.npz",
    "../Data/bands/combined_15ch_full_day_no_twillight.npz",
    "../Data/bands/combined_15ch_full_night_twillight.npz",
    day_thresh = 80.0,
    night_thresh = 90.0,
    twilight="night",          # strict day/night only
    drop_nan_timesteps=True,  # remove timesteps that become all-NaN 
)

In [ ]:
day_bands = np.load("../Data/bands/combined_15ch_full_day_no_twillight.npz")
day_data = day_bands["data"]
day_time = day_bands["time_min"]

## 3. Temporal Alignment
The segmented satellite bands are then paired with CAMS irradiance data (`ghi_grid2500_2ch_15mins.npz`, comprising 140,214 initial timestamps). Because sensor readings do not always perfectly align, a fuzzy matching algorithm is applied with a maximum tolerance of 6 minutes.


### Daytime Dataset Pairing

In [ ]:
# load CAMS data
cams = np.load("../Data/CAMS/ghi_grid2500_2ch_15mins.npz")
cams_data = cams["data"]
cams_time = cams["time"]

# pair CAMS and bands data for daytime only
pair_and_save_cams_bands(
    bands_npz_path = "../Data/bands/combined_15ch_full_day_no_twillight.npz",
    cams_npz_path = "../Data/CAMS/ghi_grid2500_2ch_15mins.npz",
    out_cams_npz_path="../Data/CAMS/modeldata/cams_day_paired_no_twillight.npz",
    out_bands_npz_path= "../Data/bands/modeldata/bands_day_paired_no_twillight.npz",
    tol_min=6,
    filter_cams = False, 
    one_to_one=False 
)

### Nighttime and Twilight Dataset Pairing

In [ ]:
# pair CAMS and bands data for nighttime only
pair_and_save_cams_bands(
    bands_npz_path = "../Data/bands/combined_15ch_full_night_twillight.npz",
    cams_npz_path = "../Data/CAMS/ghi_grid2500_2ch_15mins.npz",
    out_cams_npz_path="../Data/CAMS/modeldata/cams_night_paired_twillight.npz",
    out_bands_npz_path= "Data/bands/modeldata/bands_night_paired_twillight.npz",
    tol_min=6,
    filter_cams = False, 
    one_to_one=True
)

In [5]:
import numpy as np
import pandas as pd

def prepare_night_bands_from_cams_memory_safe(
    path_cams: str,
    path_bands: str,
    out_night_path: str,
    atol: float = 1e-3,
    eps: float = 1e-6,
    tolerance_min: int = 6,
    drop_nan_timesteps: bool = True
) -> None:
    print("🌙 Preparing Night Bands strictly from CAMS csGHI baseline (Memory Safe Mode)...")

    # ---------------------------------------------------------
    # 1. Identify "Night" Timestamps from CAMS
    # ---------------------------------------------------------
    cams = np.load(path_cams)
    time_key = 'time_hourly' if 'time_hourly' in cams.files else 'time'
    t_cams = cams[time_key]
    data_cams = cams['data']  
    
    csghi = data_cams[:, 1, :, :]
    is_night_cams = np.all(csghi <= eps, axis=(1, 2))
    
    t_cams_night = t_cams[is_night_cams]
    dt_cams_night = pd.to_datetime(t_cams_night, utc=True)
    
    # Free up RAM immediately
    del cams, data_cams, csghi, is_night_cams
    print(f"  • Found {len(dt_cams_night)} absolute night frames in CAMS.")

    # ---------------------------------------------------------
    # 2. Match Timelines BEFORE Loading the Heavy Data
    # ---------------------------------------------------------
    z = np.load(path_bands)
    bands_time = z["time_min"].astype(np.int64, copy=False)
    lbands = z["bands"] if "bands" in z.files else np.array([])
    
    dt_bands = pd.to_datetime(bands_time, unit='m', utc=True)

    cams_df = pd.DataFrame(index=dt_cams_night).sort_index()
    bands_df = pd.DataFrame({'bands_idx': np.arange(len(dt_bands))}, index=dt_bands).sort_index()
    
    merged = pd.merge_asof(
        cams_df,
        bands_df,
        left_index=True,
        right_index=True,
        direction='nearest',
        tolerance=pd.Timedelta(minutes=tolerance_min)
    )
    
    merged_clean = merged.dropna(subset=['bands_idx'])
    matched_bands_idx = merged_clean['bands_idx'].astype(int).drop_duplicates().values
    
    night_time = bands_time[matched_bands_idx]
    print(f"  • Matched {len(matched_bands_idx)} satellite frames. Extracting data...")

    # ---------------------------------------------------------
    # 3. Extract Only the Needed Frames (RAM Saver)
    # ---------------------------------------------------------
    # We slice directly from the uncompressed z["data"] object.
    # This prevents the entire 30GB array from being loaded at once.
    bands_data_raw = z["data"]
    
    # Extract just the night frames into memory
    night_data = bands_data_raw[matched_bands_idx].astype(np.float32)
    
    # Free the original file handle
    z.close()

    # ---------------------------------------------------------
    # 4. Sanitize Only the Extracted Night Data
    # ---------------------------------------------------------
    print("  • Sanitizing global minimums to NaN...")
    global_min = float(np.nanmin(night_data))
    night_data[np.isclose(night_data, global_min, atol=atol)] = np.nan

    if drop_nan_timesteps:
        valid_frames = ~np.all(np.isnan(night_data), axis=tuple(range(1, night_data.ndim)))
        night_data = night_data[valid_frames]
        night_time = night_time[valid_frames]
        print(f"  • Dropped {len(matched_bands_idx) - len(valid_frames)} all-NaN timesteps.")

    # ---------------------------------------------------------
    # 5. Save Final Subset
    # ---------------------------------------------------------
    np.savez_compressed(
        out_night_path, 
        data=night_data, 
        time_min=night_time, 
        bands=lbands
    )
    
    print(f"✅ Saved correctly aligned Night Bands to: {out_night_path}")
    print(f"  • Final Shape: {night_data.shape}")

In [ ]:
import numpy as np
import pandas as pd

# 1. Load the original CAMS dataset
path_cams = "Data/CAMS/ghi_grid2500_2ch_15mins.npz" # Use your actual 2ch or 3ch CAMS path
cams = np.load(path_cams)

time_key = 'time_hourly' if 'time_hourly' in cams.files else 'time'
t_cams = cams[time_key]
data_cams = cams['data']

# 2. Identify frames where csGHI (Channel 1) is effectively 0 across the entire grid
eps = 1e-6
csghi = data_cams[:, 1, :, :]
is_zero_csghi = np.all(csghi <= eps, axis=(1, 2))

# 3. Convert timeline to HKT
dt_utc = pd.to_datetime(t_cams, utc=True)
dt_hkt = dt_utc.tz_convert('Asia/Hong_Kong')

# 4. Build a DataFrame and Group by Hour
df_stats = pd.DataFrame({
    'Hour (HKT)': dt_hkt.hour,
    'Is_Zero_csGHI': is_zero_csghi
})

grouped = df_stats.groupby('Hour (HKT)').agg(
    Zero_csGHI_Frames=('Is_Zero_csGHI', 'sum'),
    Total_Frames=('Is_Zero_csGHI', 'count')
)

grouped['Zero_Percentage'] = (grouped['Zero_csGHI_Frames'] / grouped['Total_Frames']) * 100

print("="*60)
print("🔍 CAMS csGHI==0 HOURLY DISTRIBUTION (HKT)")
print("="*60)
print(grouped)
print("="*60)

In [6]:
prepare_night_bands_from_cams_memory_safe(
    path_cams="Data/CAMS/ghi_grid2500_2ch_15mins.npz",
    path_bands="Data/bands/combined_15ch_full.npz",
    out_night_path="Data/bands/modeldata/bands_night_paired_zero_frames.npz",
    tolerance_min=6,
)

🌙 Preparing Night Bands strictly from CAMS csGHI baseline (Memory Safe Mode)...
  • Found 68230 absolute night frames in CAMS.
  • Matched 50901 satellite frames. Extracting data...
  • Sanitizing global minimums to NaN...
  • Dropped 0 all-NaN timesteps.
✅ Saved correctly aligned Night Bands to: Data/bands/modeldata/bands_night_paired_zero_frames.npz
  • Final Shape: (50846, 15, 50, 50)
